In [ ]:
specTESS = {
  "seed": 183074220,
  "koAngles_SolarPanel":[56.0,124.0],
  "missionLife": 3,
  "checkKeepoutEnd": True,
  "pupilDiam": 2.37,
  "scienceInstruments": [
    { "name": "imager"
    }
  ],
  "starlightSuppressionSystems": [
    { "name": "HLC-565",
      "koAngles_Sun":[45.0,180.0],
      "koAngles_Earth":[45.0,180.0],
      "koAngles_Moon":[45.0,180.0],
      "koAngles_Small":[1.0,180.0]
    }
  ],
  "observingModes": [
    { "instName": "imager",
      "systName": "HLC-565",
      "detectionMode": True,
      "SNR": 5
    }
  ],
    "modules": {
        "PlanetPopulation": "SAG13",
        "StarCatalog": "EXOCAT1",
        "OpticalSystem": " ",
        "ZodiacalLight": " ",
        "BackgroundSources": " ",
        "PlanetPhysicalModel": " ",
        "Observatory": "ObservatoryTESScsv",
        "TimeKeeping": " ",
        "PostProcessing": " ",
        "Completeness": "BrownCompleteness",
        "TargetList": " ",
        "SimulatedUniverse": "SAG13Universe",
        "SurveySimulation": " ",
        "SurveyEnsemble": " ",
    }
}

# initialize mission sim
simTESS = EXOSIMS.MissionSim.MissionSim(**copy.deepcopy(specTESS))
OSTESS = simTESS.OpticalSystem
obsTESS, TLTESS, TKTESS = simTESS.Observatory, simTESS.TargetList, simTESS.TimeKeeping

# time array
koEvaltimesTESS = Time(
    np.arange(TKTESS.missionStart.value,
              TKTESS.missionStart.value+TKTESS.missionLife.to('day').value,
              1),
    format='mjd'
)

# construct koangles
systNamesTESS = np.unique([OSTESS.observingModes[x]['syst']['name']
                       for x in np.arange(len(OSTESS.observingModes))])
koStrTESS     = ["koAngles_Sun", "koAngles_Moon", "koAngles_Earth", "koAngles_Small"]
koanglesTESS  = np.zeros([len(systNamesTESS),4,2])
for x in np.argsort(systNamesTESS):
    rel_modeTESS = list(filter(lambda mode: mode['syst']['name'] == systNamesTESS[x], OSTESS.observingModes))[0]
    koanglesTESS[x] = np.asarray([rel_modeTESS['syst'][k] for k in koStrTESS])

# storage arrays
NstarsTESS = min(3, len(TLTESS.Name))  # ensures you don't exceed total number of stars
selected_indicesTESS = np.arange(NstarsTESS)
NtimesTESS = len(koEvaltimesTESS)
kogoodTESS  = np.zeros((NstarsTESS, NtimesTESS), dtype=bool)
culpritTESS = np.zeros((NstarsTESS, NtimesTESS, 12), dtype=bool)

# loop over all stars
for idx, i in enumerate(selected_indicesTESS):
    for t, date in enumerate(koEvaltimesTESS):
        tmpkogoodTESS, r_bodyTESS, r_targTESS, tmpculpritTESS, koangleArrayTESS = obsTESS.keepout(
            TLTESS, [i, i], date, koanglesTESS, True
        )
        kogoodTESS[idx, t]   = tmpkogoodTESS[0,0,0]
        culpritTESS[idx, t,:] = tmpculpritTESS[0,0,0,:]

In [ ]:
cmap = colors.ListedColormap(
   ["white", "#FFD500", "blue", "#747783", "red", "m", "red"]
)  # colors for used to indicate a culprit behind keepout
bounds = [0, 1, 2, 3, 4, 5, 6, 7]
norm = colors.BoundaryNorm(bounds, cmap.N)

# creating an array of colors based on culprit
koColor = np.zeros([TLTESS.nStars, len(koEvaltimesTESS)])
koColor = (
    culpritTESS[:, :, 0] * 1
    + culpritTESS[:, :, 2] * 2
    + culpritTESS[:, :, 1] * 3
    + culpritTESS[:, :, 3] * 6
    + culpritTESS[:, :, 4] * 5
    + culpritTESS[:, :, 5] * 4
    + culpritTESS[:, :, 11] * 1
)
koColor.astype("int")


# plotting colors on a 2d map
plt.close(546832183)
fig = plt.figure(546832183, figsize=(10, 5))
fig.subplots_adjust(bottom=0.15)
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], height_ratios=[1])
gs.update(wspace=0.06, hspace=0.06)  # set the spacing between axes.
plt.rc("axes", linewidth=2)
plt.rc("lines", linewidth=2)
plt.rcParams["axes.linewidth"] = 2
plt.rc("font", weight="bold")
ax2 = plt.subplot(gs[1])
ax = plt.subplot(gs[0])


if (
    koColor.shape[0] > 100
):  # Determine maximum number of stars to track keepouts for
    NUMBER_Y = 100
else:
    NUMBER_Y = koColor.shape[0]
sIndsTESS = np.linspace(
    0, koColor.shape[0], num=NUMBER_Y, endpoint=False, dtype=int
).tolist()
img = plt.imshow(
    koColor[sIndsTESS, 0 : int(np.floor(len(koEvaltimesTESS)))],
    aspect="auto",  # 4,
    cmap=cmap,
    interpolation="none",
    origin="lower",
    norm=norm,
)


ax.set_xlabel(
    "Mission Elapsed Time (d), Mission Start %s UTC MJD"
    % (str(TKTESS.missionStart.value)),
    weight="bold",
)
ax.set_ylabel(r"Target Star, $i$", weight="bold")
ax.set_xlim(left=0.0, right=np.max(koEvaltimesTESS).value - TKTESS.missionStart.value)
ax.set_ylim(bottom=0.0, top=NUMBER_Y)

outline = PathEffects.withStroke(linewidth=5, foreground="black")
plt.plot(
    [-1.0, -1.0],
    [-1.0, -1.0],
    color=cmap.colors[0],
    label="Visible",
    path_effects=[outline],
    )
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[1], label="\u2609")
plt.plot(
    [-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[2], label=r"$\oplus$"
)  ##\u2641$')
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[3], label="\u263D")
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[4], label="\u2642\u263F")
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[5], label="\u2640")
# plt.plot([-1.,-1.],[-1.,-1.],color=cmap.colors[6],label=ur'$\u263F$')  duplicate color so appended above
leg = plt.legend(framealpha=1.0)
# get the lines and texts inside legend box
leg_lines = leg.get_lines()
leg_texts = leg.get_texts()
# bulk-set the properties of all lines and texts
plt.setp(leg_lines, linewidth=4)
plt.setp(leg_texts, fontsize="x-large")
nullfmt = NullFormatter()
ax2.yaxis.set_major_formatter(nullfmt)

# Plot horizontal histogram
tTotal = (
    np.max(koEvaltimesTESS).value - TKTESS.missionStart.value
)  # Calculate total time width of koMap
tVisTESS = list()  # stores time visible of each star
for i in np.arange(
    len(sIndsTESS)
):  # iterate over all stars and append amount of time each star is visible
    tVisTESS.append(len(np.where(koColor[sIndsTESS[i], :] == 0)[0]))

widthTESS = np.zeros(len(tVisTESS)) + 1.0
ax2.barh(
    np.arange(len(sIndsTESS)) + 0.5,
    np.asarray(tVisTESS, dtype=float) / tTotal * 100.0,
    widthTESS,
    align="center",
    color="black",
)
ax2.set_xlim(left=0.0, right=100.0)
ax2.set_ylim(bottom=0.0, top=NUMBER_Y)
ax2.set_xlabel("% Time\n Visible", weight="bold")
#plt.show(block=False)
plt.show()
plt.gcf().canvas.draw()

DT = datetime.datetime
date = str(DT.now())  # ,"utf-8")
date = "".join(
    c + "_" for c in re.split("-|:| ", date)[0:-1]
)  # Removes seconds from date

In [ ]:
# TESS Histogram Percent Time Visible
plt.close(98735654)
fig = plt.figure(98735654)
binsTESS = np.linspace(
    start=0,
    stop=np.round(np.max(tVisTESS) / tTotal * 100.0),
    num=int(np.round(np.max(tVisTESS) / tTotal * 100.0) + 1),
)
plt.hist(
    np.asarray(tVisTESS) / tTotal * 100.0,
    bins=binsTESS,
    color="black",
    alpha=1.0,
    histtype="bar",
    ec="black",
)
plt.ylabel("Target Count", weight="bold")
plt.xlabel("Time Visible (%)", weight="bold")
plt.xlim((0, np.ceil(np.max(tVisTESS) / tTotal * 100.0)))
#plt.show(block=False)
plt.show()
plt.gcf().canvas.draw()

DT = datetime.datetime
date = str(DT.now())  # ,"utf-8")
date = "".join(
    c + "_" for c in re.split("-|:| ", date)[0:-1]
)  # Removes seconds from date

In [ ]:
specKep = {
  "seed": 183074220,
  "koAngles_SolarPanel":[56.0,124.0],
  "missionLife": 3,
  "checkKeepoutEnd": True,
  "pupilDiam": 2.37,
  "scienceInstruments": [
    { "name": "imager"
    }
  ],
  "starlightSuppressionSystems": [
    { "name": "HLC-565",
      "koAngles_Sun":[45.0,180.0],
      "koAngles_Earth":[45.0,180.0],
      "koAngles_Moon":[45.0,180.0],
      "koAngles_Small":[1.0,180.0]
    }
  ],
  "observingModes": [
    { "instName": "imager",
      "systName": "HLC-565",
      "detectionMode": True,
      "SNR": 5
    }
  ],
    "modules": {
        "PlanetPopulation": "SAG13",
        "StarCatalog": "EXOCAT1",
        "OpticalSystem": " ",
        "ZodiacalLight": " ",
        "BackgroundSources": " ",
        "PlanetPhysicalModel": " ",
        "Observatory": "ObservatoryKepHalo",
        "TimeKeeping": " ",
        "PostProcessing": " ",
        "Completeness": "BrownCompleteness",
        "TargetList": " ",
        "SimulatedUniverse": "SAG13Universe",
        "SurveySimulation": " ",
        "SurveyEnsemble": " ",
    }
}

# initialize mission sim
simKep = EXOSIMS.MissionSim.MissionSim(**copy.deepcopy(specKep))
OSKep = simKep.OpticalSystem
obsKep, TLKep, TKKep = simKep.Observatory, simKep.TargetList, simKep.TimeKeeping

# time array
koEvaltimesKep = Time(
    np.arange(TKKep.missionStart.value,
              TKKep.missionStart.value+TKKep.missionLife.to('day').value,
              1),
    format='mjd'
)

# construct koangles
systNamesKep = np.unique([OSKep.observingModes[x]['syst']['name']
                       for x in np.arange(len(OSKep.observingModes))])
koStrKep     = ["koAngles_Sun", "koAngles_Moon", "koAngles_Earth", "koAngles_Small"]
koanglesKep  = np.zeros([len(systNamesKep),4,2])
for x in np.argsort(systNamesKep):
    rel_modeKep = list(filter(lambda mode: mode['syst']['name'] == systNamesKep[x], OSKep.observingModes))[0]
    koanglesKep[x] = np.asarray([rel_modeKep['syst'][k] for k in koStrKep])

# storage arrays
NstarsKep = min(3, len(TLKep.Name))  # ensures you don't exceed total number of stars
selected_indicesKep = np.arange(NstarsKep)
NtimesKep = len(koEvaltimesKep)
kogoodKep  = np.zeros((NstarsKep, NtimesKep), dtype=bool)
culpritKep = np.zeros((NstarsKep, NtimesKep, 12), dtype=bool)

# loop over all stars
for idx, i in enumerate(selected_indicesKep):
    for t, date in enumerate(koEvaltimesKep):
        tmpkogoodKep, r_bodyKep, r_targKep, tmpculpritKep, koangleArrayKep = obsKep.keepout(
            TLKep, [i, i], date, koanglesKep, True
        )
        kogoodKep[idx, t]   = tmpkogoodKep[0,0,0]
        culpritKep[idx, t,:] = tmpculpritKep[0,0,0,:]

In [ ]:
# Kep Halo Keepout Map Colors
cmap = colors.ListedColormap(
   ["white", "#FFD500", "blue", "#747783", "red", "m", "red"]
)  # colors for used to indicate a culprit behind keepout
bounds = [0, 1, 2, 3, 4, 5, 6, 7]
norm = colors.BoundaryNorm(bounds, cmap.N)

# creating an array of colors based on culprit
koColor = np.zeros([TLKep.nStars, len(koEvaltimesKep)])
koColor = (
    culpritKep[:, :, 0] * 1
    + culpritKep[:, :, 2] * 2
    + culpritKep[:, :, 1] * 3
    + culpritKep[:, :, 3] * 6
    + culpritKep[:, :, 4] * 5
    + culpritKep[:, :, 5] * 4
    + culpritKep[:, :, 11] * 1
)
koColor.astype("int")


# plotting colors on a 2d map
plt.close(546832183)
fig = plt.figure(546832183, figsize=(10, 5))
fig.subplots_adjust(bottom=0.15)
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], height_ratios=[1])
gs.update(wspace=0.06, hspace=0.06)  # set the spacing between axes.
plt.rc("axes", linewidth=2)
plt.rc("lines", linewidth=2)
plt.rcParams["axes.linewidth"] = 2
plt.rc("font", weight="bold")
ax2 = plt.subplot(gs[1])
ax = plt.subplot(gs[0])


if (
    koColor.shape[0] > 100
):  # Determine maximum number of stars to track keepouts for
    NUMBER_Y = 100
else:
    NUMBER_Y = koColor.shape[0]
sIndsKep = np.linspace(
    0, koColor.shape[0], num=NUMBER_Y, endpoint=False, dtype=int
).tolist()
img = plt.imshow(
    koColor[sIndsKep, 0 : int(np.floor(len(koEvaltimesKep)))],
    aspect="auto",  # 4,
    cmap=cmap,
    interpolation="none",
    origin="lower",
    norm=norm,
)


ax.set_xlabel(
    "Mission Elapsed Time (d), Mission Start %s UTC MJD"
    % (str(TKKep.missionStart.value)),
    weight="bold",
)
ax.set_ylabel(r"Target Star, $i$", weight="bold")
ax.set_xlim(left=0.0, right=np.max(koEvaltimesKep).value - TKKep.missionStart.value)
ax.set_ylim(bottom=0.0, top=NUMBER_Y)

outline = PathEffects.withStroke(linewidth=5, foreground="black")
plt.plot(
    [-1.0, -1.0],
    [-1.0, -1.0],
    color=cmap.colors[0],
    label="Visible",
    path_effects=[outline],
    )
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[1], label="\u2609")
plt.plot(
    [-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[2], label=r"$\oplus$"
)  ##\u2641$')
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[3], label="\u263D")
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[4], label="\u2642\u263F")
plt.plot([-1.0, -1.0], [-1.0, -1.0], color=cmap.colors[5], label="\u2640")
# plt.plot([-1.,-1.],[-1.,-1.],color=cmap.colors[6],label=ur'$\u263F$')  duplicate color so appended above
leg = plt.legend(framealpha=1.0)
# get the lines and texts inside legend box
leg_lines = leg.get_lines()
leg_texts = leg.get_texts()
# bulk-set the properties of all lines and texts
plt.setp(leg_lines, linewidth=4)
plt.setp(leg_texts, fontsize="x-large")
nullfmt = NullFormatter()
ax2.yaxis.set_major_formatter(nullfmt)

# Plot horizontal histogram
tTotal = (
    np.max(koEvaltimesKep).value - TKKep.missionStart.value
)  # Calculate total time width of koMap
tVisKep = list()  # stores time visible of each star
for i in np.arange(
    len(sIndsKep)
):  # iterate over all stars and append amount of time each star is visible
    tVisKep.append(len(np.where(koColor[sIndsKep[i], :] == 0)[0]))

widthKep = np.zeros(len(tVisKep)) + 1.0
ax2.barh(
    np.arange(len(sIndsKep)) + 0.5,
    np.asarray(tVisKep, dtype=float) / tTotal * 100.0,
    widthKep,
    align="center",
    color="black",
)
ax2.set_xlim(left=0.0, right=100.0)
ax2.set_ylim(bottom=0.0, top=NUMBER_Y)
ax2.set_xlabel("% Time\n Visible", weight="bold")
#plt.show(block=False)
plt.show()
plt.gcf().canvas.draw()

DT = datetime.datetime
date = str(DT.now())  # ,"utf-8")
date = "".join(
    c + "_" for c in re.split("-|:| ", date)[0:-1]
)  # Removes seconds from date

In [ ]:
# Kep Histogram Percent Time Visible
plt.close(98735654)
fig = plt.figure(98735654)
binsKep = np.linspace(
    start=0,
    stop=np.round(np.max(tVisKep) / tTotal * 100.0),
    num=int(np.round(np.max(tVisKep) / tTotal * 100.0) + 1),
)
plt.hist(
    np.asarray(tVisKep) / tTotal * 100.0,
    bins=binsKep,
    color="black",
    alpha=1.0,
    histtype="bar",
    ec="black",
)
plt.ylabel("Target Count", weight="bold")
plt.xlabel("Time Visible (%)", weight="bold")
plt.xlim((0, np.ceil(np.max(tVisKep) / tTotal * 100.0)))
#plt.show(block=False)
plt.show()
plt.gcf().canvas.draw()

DT = datetime.datetime
date = str(DT.now())  # ,"utf-8")
date = "".join(
    c + "_" for c in re.split("-|:| ", date)[0:-1]
)  # Removes seconds from date